# Eksperimen 05: Stacking Ensemble (Fase M1 & M2)


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

os.environ['PYTHONHASHSEED'] = '0'
np.random.seed(42)

if 'train_df' not in locals():
    print('Memuat data...')
    train_df = pd.read_parquet('../../dataset/cleaned_dataset.parquet')
    print(f'Ukuran awal: {train_df.shape}')


## M1: Post-Preprocessing EDA


In [ ]:
if 'train_df' in locals():
    dups = train_df.duplicated(subset=['movie_title', 'cinema_ids', 'date_show']).sum()
    print(f'Duplikasi: {dups}')
    if dups > 0:
        print('Menghapus duplikasi...')
        train_df.drop_duplicates(subset=['movie_title', 'cinema_ids', 'date_show'], keep='first', inplace=True)

    if not pd.api.types.is_datetime64_any_dtype(train_df['date_show']):
        train_df['date_show'] = pd.to_datetime(train_df['date_show'])
    print('Tahap M1 Selesai')


## M1: Visualisasi Distribusi Tiket Awal
Tujuan: Memastikan tidak ada agregasi ganda yang lolos sebelum proses Sliding Window.

In [ ]:
if 'train_df' in locals():
    plt.figure(figsize=(10, 5))
    sns.histplot(train_df['total_ticket'], bins=50, kde=True, color='teal')
    plt.title('Distribusi Total Tiket per Hari (Sebelum Sliding Window)')
    plt.xlim(0, train_df['total_ticket'].quantile(0.99))
    plt.xlabel('Total Ticket')
    plt.ylabel('Frekuensi')
    plt.show()


## M2: Rekayasa Fitur Hibrida (Cuaca, Sineas, Kanibalisme)


In [ ]:
if 'train_df' in locals():
    print('Mengekstrak fitur...')
    if 'total_show' in train_df.columns:
        train_df.drop(columns=['total_show'], inplace=True)

    # Proteksi jika cell di-run berkali-kali
    train_df.drop(columns=['rainfall_mm', 'is_heavy_rain', 'weather_penalty_score', 'rain_intensity_impact', 'is_squeezed'], inplace=True, errors='ignore')

    weather_df = pd.read_csv('../../dataset/weather_data.csv')
    weather_df['date_show'] = pd.to_datetime(weather_df['date_show'])
    train_df = train_df.merge(weather_df, on=['city_name', 'date_show'], how='left')
    train_df['rainfall_mm'] = train_df['rainfall_mm'].fillna(0)
    train_df['is_heavy_rain'] = train_df['is_heavy_rain'].fillna(0)

    train_df['weather_penalty_score'] = train_df['is_heavy_rain'] * train_df['is_weekend']
    train_df['rain_intensity_impact'] = train_df['is_heavy_rain'] * train_df['rainfall_mm']
    train_df['is_squeezed'] = ((train_df['screen_share_pct'] < 15) & (train_df['is_clashing_blockbuster'] == 1)).astype('int8')

    if 'release_date' in train_df.columns:
        train_df['release_date'] = pd.to_datetime(train_df['release_date'])
        train_df['day_since_release'] = (train_df['date_show'] - train_df['release_date']).dt.days + 1
    else:
        first_show = train_df.groupby('movie_title')['date_show'].min().reset_index()
        first_show.rename(columns={'date_show': 'inferred_release_date'}, inplace=True)
        train_df = train_df.merge(first_show, on='movie_title', how='left')
        train_df['day_since_release'] = (train_df['date_show'] - train_df['inferred_release_date']).dt.days + 1
    print('Fitur berhasil ditambahkan')


## M2: Transformasi Jendela Waktu (D1-D3) & Pre-Stacking Prep


In [ ]:
if 'train_df' in locals():
    print('Memulai perakitan jendela waktu (ini mungkin butuh beberapa detik)...')
    train_df.sort_values(by=['movie_title', 'cinema_ids', 'date_show'], inplace=True)
    train_df['day_idx'] = train_df.groupby(['movie_title', 'cinema_ids']).cumcount() + 1

    pair_counts = train_df.groupby(['movie_title', 'cinema_ids']).size()
    valid_pairs = pair_counts[pair_counts >= 4].index
    df_valid = train_df.set_index(['movie_title', 'cinema_ids']).loc[valid_pairs].reset_index()

    history = df_valid[df_valid['day_idx'] <= 3]
    history_agg = history.groupby(['movie_title', 'cinema_ids'], observed=True).agg(
        d1_ticket=('total_ticket', lambda x: x.iloc[0] if len(x)>0 else np.nan),
        d2_ticket=('total_ticket', lambda x: x.iloc[1] if len(x)>1 else np.nan),
        d3_ticket=('total_ticket', lambda x: x.iloc[2] if len(x)>2 else np.nan),
        scale_sp=('total_ticket', lambda x: max(x.mean(), 1))
    ).reset_index()

    history_agg['D3_D1_ratio'] = history_agg['d3_ticket'] / history_agg['d1_ticket'].replace(0, 1)
    p99 = history_agg['D3_D1_ratio'].quantile(0.99)
    history_agg['D3_D1_ratio'] = np.clip(history_agg['D3_D1_ratio'], a_min=0, a_max=p99)

    targets = df_valid[(df_valid['day_idx'] >= 4) & (df_valid['day_idx'] <= 10)]
    train_final = targets.merge(history_agg, on=['movie_title', 'cinema_ids'], how='left')

    for col in ['movie_title', 'cinema_ids', 'city_name', 'genre', 'age_rating']:
        if col in train_final.columns:
            train_final[col] = train_final[col].astype('category')
            
    print(f'Dimensi data final (siap latih): {train_final.shape}')
    display(train_final[['movie_title', 'day_since_release', 'total_ticket', 'scale_sp', 'D3_D1_ratio', 'weather_penalty_score']].head())


## M2: Visualisasi Efektivitas Fitur Hibrida (Ablation Pre-check)
Tujuan: Memverifikasi apakah cuaca ekstrim dan kanibalisme blockbuster benar-benar merusak penjualan tiket di hari tayang, serta menganalisis pola eksponensial peluruhan.

In [ ]:
if 'train_final' in locals():
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # 1. Efek Cuaca Buruk
    sns.boxplot(x='weather_penalty_score', y='total_ticket', data=train_final, ax=axes[0], palette='Blues')
    axes[0].set_title('Efek Cuaca Akhir Pekan (Penalty) vs Tiket D4-D10')
    axes[0].set_ylim(0, 500)
    axes[0].set_xlabel('Weather Penalty (0=Aman, 1=Hujan Lebat)')
    
    # 2. Efek Kanibalisme
    sns.boxplot(x='is_squeezed', y='total_ticket', data=train_final, ax=axes[1], palette='Reds')
    axes[1].set_title('Kanibalisme Blockbuster vs Tiket D4-D10')
    axes[1].set_ylim(0, 500)
    axes[1].set_xlabel('Squeezed (0=Bebas, 1=Tergencet)')
    
    # 3. Distribusi Rasio D3/D1
    sns.histplot(train_final['D3_D1_ratio'].dropna(), bins=40, kde=True, ax=axes[2], color='orange')
    axes[2].set_title('Distribusi Rasio Peluruhan (D3/D1 Ratio)')
    axes[2].set_xlabel('D3/D1 Ratio (Clipped)')
    
    plt.tight_layout()
    plt.show()
